# PlantSeg: extra inference

Requires the trained models from `01_train_colab.ipynb` on Drive (`ROOT/model/<run>.pt`) and the repo's `src/` under `ROOT/src`.
Select runtime type **A100 GPU** → *Runtime → Run all*. No training; test inference with finished models only (~45-60 min).
Output on Drive `ROOT/results/extra/`: class-agnostic mask IoU (`<run>_test_fg.npz`) and prediction maps for 18 example images (`predictions/`).
If interrupted, *Run all* again: finished models are skipped.

In [ ]:
# 1) Settings
ROOT = '/content/drive/MyDrive/plantseg-dinov3-lora'
RUNS = 'a_v3L_lora_s0,d_v3L_lora_s1,e_v3L_lora_s2,b_cnxL_s0,f_cnxL_s1,g_cnxL_s2,c_cnxL_mak_s0,h_v2L_lora_s0,i_H2_s0'
VISUAL = 'coffee_leaf_rust_google_0135,grape_downy_mildew_251,wheat_loose_smut_Bing_0178,soybean_mosaic_Bing_0010,apple_black_rot_7,tomato_yellow_leaf_curl_virus_44,bell_pepper_bacterial_spot_Bing_0008,rice_sheath_blight_148,lettuce_mosaic_virus_8,strawberry_leaf_scorch_google_0186,cherry_powdery_mildew_76,eggplant_phytophthora_blight_Google_0028,soybean_mosaic_Bing_0076,coffee_berry_blotch_Bing_0027,celery_early_blight_9,cucumber_powdery_mildew_59,cucumber_powdery_mildew_google_0000,cucumber_angular_leaf_spot_9'  # 12 clean test images (head/mid/tail × 4, seed 0) + 3 each where DINOv3 leads most and least
AUTO_DISCONNECT = True
REQUIRE_A100 = True   # main results were obtained with bf16/A100; same environment

In [ ]:
# 2) Drive + GPU + code
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess
assert os.path.isfile(f'{ROOT}/src/extra_inference.py'), 'src/extra_inference.py missing on Drive'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
if REQUIRE_A100 and 'A100' not in gpu:
    raise SystemExit('Not an A100: set the runtime type to A100 (or REQUIRE_A100=False).')
shutil.copytree(f'{ROOT}/src', '/content/src', dirs_exist_ok=True)
os.environ['HF_HOME'] = '/content/hf'

In [ ]:
# 3) Dependencies
!pip -q install timm==1.0.30 imagehash==4.3.2
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)

In [ ]:
# 4) Data (train/val/test preparation) + inference
!python /content/src/extra_inference.py --root "{ROOT}" --target /content/plantseg --runs "{RUNS}" --visual "{VISUAL}" --yes_test

In [ ]:
# 5) Check + disconnect the runtime
print(sorted(os.listdir(f'{ROOT}/results/extra')))
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()